# Pertemuan 3 — Data Preparation & KNN Imputation

Notebook ini berisi:
1. **Upload & tampilkan gambar** dari Orange/PostgreSQL
2. **Load dataset** Iris dan Bank
3. **Identifikasi missing value** dan statistik deskriptif
4. **KNN Imputation** pada data numerik (Iris) — perhitungan manual + kode
5. **KNN Imputation** pada data campuran (Bank) — perhitungan manual + kode
6. **Visualisasi** tabel jarak dan hasil imputasi

---

## Petunjuk Penggunaan di Google Colab

Sebelum menjalankan notebook ini, upload file-file berikut:

**Dataset:**
- `IRIS.csv` — dataset Iris (150 baris)
- `bank.csv` — dataset Bank Marketing (239 baris)

**Gambar dari folder `Assets/Pertemuan3/`:**
- `DataIrisOrangePengukuranJarak.png`
- `Gambar Csv ke PostgreeSQL.png`
- `SebelumScalling.png`
- `SesudahScalling.png`

Jalankan cell di bawah untuk mengupload semua file sekaligus.

---
## 1. Upload File yang Dibutuhkan

In [3]:
from google.colab import files
import os

print("=" * 60)
print("UPLOAD FILE")
print("=" * 60)
print("Silakan upload file berikut (bisa pilih sekaligus):")
print("  1. IRIS.csv")
print("  2. bank.csv")
print("  3. DataIrisOrangePengukuranJarak.png")
print("  4. Gambar Csv ke PostgreeSQL.png")
print("  5. SebelumScalling.png")
print("  6. SesudahScalling.png")
print()

uploaded = files.upload()

print("\nFile yang berhasil diupload:")
for fname in uploaded.keys():
    print(f"  ✓ {fname} ({len(uploaded[fname])} bytes)")

ModuleNotFoundError: No module named 'google'

In [ ]:
# Cek semua file yang dibutuhkan sudah ada
required_files = {
    'dataset': ['IRIS.csv', 'bank.csv'],
    'gambar': [
        'DataIrisOrangePengukuranJarak.png',
        'Gambar Csv ke PostgreeSQL.png',
        'SebelumScalling.png',
        'SesudahScalling.png'
    ]
}

print("Cek file yang tersedia:")
all_ok = True
for kategori, flist in required_files.items():
    print(f"\n{kategori.upper()}:")
    for f in flist:
        exists = os.path.exists(f)
        status = "✓" if exists else "✗ BELUM ADA"
        print(f"  {status} {f}")
        if not exists:
            all_ok = False

if all_ok:
    print("\n✅ Semua file sudah lengkap! Lanjutkan ke cell berikutnya.")
else:
    print("\n⚠️ Ada file yang belum diupload. Jalankan ulang cell upload di atas.")

---
## 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

print("Library berhasil di-import.")

---
## 3. Menampilkan Gambar dari Orange & PostgreSQL

Gambar-gambar berikut adalah bukti dari workflow Orange Data Mining dan hasil import data ke PostgreSQL.

### 3.1 Workflow Pengukuran Jarak Dataset Iris di Orange

Workflow ini menunjukkan alur pengukuran jarak di Orange:
- **CSV File Import** → memuat data Iris
- **Data Table** → menampilkan isi data
- Widget jarak: **Euclidean**, **Manhattan**, **Spearman**, **Hamming**
- **Distance Matrix** → menampilkan matriks jarak
- **Save Distance Matrix** → menyimpan hasil

In [ ]:
# Tampilkan gambar workflow Orange
if os.path.exists('DataIrisOrangePengukuranJarak.png'):
    img = plt.imread('DataIrisOrangePengukuranJarak.png')
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Workflow Pengukuran Jarak Dataset Iris di Orange', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ File DataIrisOrangePengukuranJarak.png belum diupload.")

### 3.2 Bukti Data CSV ke PostgreSQL

Dataset Iris telah berhasil dimasukkan ke PostgreSQL menggunakan query:
```sql
SELECT * FROM public.iris
```
Total data: **150 baris**.

In [ ]:
# Tampilkan bukti data di PostgreSQL
if os.path.exists('Gambar Csv ke PostgreeSQL.png'):
    img = plt.imread('Gambar Csv ke PostgreeSQL.png')
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Bukti Data CSV Telah Dimasukkan ke PostgreSQL', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ File 'Gambar Csv ke PostgreeSQL.png' belum diupload.")

### 3.3 Histogram Fitur Iris — Sebelum dan Sesudah Scaling

**Sebelum scaling:** skala antar fitur berbeda-beda (misal petal_length 1-7, sepal_width 2-4).

**Sesudah scaling (StandardScaler):** semua fitur dinormalisasi dengan mean ≈ 0 dan std ≈ 1, sehingga skala setara untuk perhitungan jarak.

In [ ]:
# Tampilkan histogram sebelum dan sesudah scaling
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Sebelum Scaling
if os.path.exists('SebelumScalling.png'):
    img1 = plt.imread('SebelumScalling.png')
    axes[0].imshow(img1)
    axes[0].set_title('Sebelum Scaling', fontsize=13, fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'File belum diupload', ha='center', va='center', fontsize=14)
    axes[0].set_title('Sebelum Scaling (belum tersedia)', fontsize=13)
axes[0].axis('off')

# Sesudah Scaling
if os.path.exists('SesudahScalling.png'):
    img2 = plt.imread('SesudahScalling.png')
    axes[1].imshow(img2)
    axes[1].set_title('Sesudah Scaling (StandardScaler)', fontsize=13, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'File belum diupload', ha='center', va='center', fontsize=14)
    axes[1].set_title('Sesudah Scaling (belum tersedia)', fontsize=13)
axes[1].axis('off')

plt.suptitle('Perbandingan Histogram Fitur Iris: Sebelum vs Sesudah Scaling', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Load Dataset Iris

In [ ]:
iris = pd.read_csv('IRIS.csv')
print(f"Dataset Iris: {iris.shape[0]} baris, {iris.shape[1]} kolom")
print(f"Kolom: {list(iris.columns)}")
iris.head(10)

---
## 5. Identifikasi Missing Value & Statistik Deskriptif — Iris

In [ ]:
# Cek missing value
print("Missing value per kolom:")
print(iris.isnull().sum())
print(f"\nTotal missing value: {iris.isnull().sum().sum()}")
print("\n→ Dataset Iris TIDAK memiliki missing value.")
print("  Oleh karena itu, akan dibuat missing value BUATAN untuk simulasi KNN.")

In [ ]:
# Grafik cek missing value Iris
import os
os.makedirs('Assets/Pertemuan3', exist_ok=True)

missing_count = iris.isnull().sum()

plt.figure(figsize=(8, 4))
missing_count.plot(kind='bar', color='steelblue')
plt.title('Jumlah Missing Value per Kolom - Iris (Sebelum Dibuat Buatan)', fontsize=12)
plt.xlabel('Kolom')
plt.ylabel('Jumlah Missing')
plt.ylim(0, 5)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/cek-missing-value-iris.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistik deskriptif Iris
numeric_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
print("Statistik Deskriptif - Dataset Iris:")
iris[numeric_cols].describe().T

In [ ]:
# Frekuensi tiap kelas
print("Frekuensi tiap species:")
print(iris['species'].value_counts())
print(f"\nTotal: {len(iris)} baris")

---
## 6. KNN Imputation — Dataset Iris (Data Numerik)

### Konsep
Dataset Iris seluruhnya numerik (kecuali `species`), sehingga jarak dihitung dengan **Euclidean Distance**.

### Langkah:
1. Buat missing value buatan pada **baris 5, kolom `petal_width`**
2. Hitung jarak Euclidean ke semua baris lain (**tanpa** kolom `petal_width`)
3. Ambil **k=3 tetangga terdekat**
4. Isi missing value = **rata-rata** `petal_width` dari 3 tetangga (KNN Regression)

### Rumus Euclidean Distance:
$$d(i,j) = \sqrt{(x_1 - y_1)^2 + (x_2 - y_2)^2 + \cdots + (x_p - y_p)^2}$$

**Aturan:** Kolom yang memiliki missing value **tidak diikutkan** dalam perhitungan jarak.

In [ ]:
# ===== MEMBUAT MISSING VALUE BUATAN =====
target_idx = 5
col_missing = 'petal_width'
nilai_asli_iris = iris.loc[target_idx, col_missing]

print(f"Data asli baris {target_idx}:")
print(iris.loc[[target_idx]])
print(f"\nNilai asli {col_missing}: {nilai_asli_iris}")

# Buat copy dan hilangkan nilai
iris_knn = iris.copy()
iris_knn.loc[target_idx, col_missing] = np.nan

print(f"\nData baris {target_idx} setelah dibuat missing:")
print(iris_knn.loc[[target_idx]])

In [ ]:
# Tabel baris yang memiliki missing value
fig, ax = plt.subplots(figsize=(10, 2))
ax.axis('off')
tabel = ax.table(
    cellText=iris_knn.loc[[target_idx]].values,
    colLabels=iris_knn.columns,
    loc='center'
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1, 1.5)
plt.title('Baris 5 dengan Missing Value Buatan (petal_width = NaN)', fontsize=11)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/baris-missing-iris.png', dpi=150, bbox_inches='tight')
plt.show()

### Perhitungan Jarak Manual (Contoh)

Karena `petal_width` kosong, jarak dihitung dari 3 kolom: `sepal_length`, `sepal_width`, `petal_length`.

**Baris 5 vs Baris 0:**
- Baris 5: [5.4, 3.9, 1.7, ?]
- Baris 0: [5.1, 3.5, 1.4]

$$d(5,0) = \sqrt{(5.4-5.1)^2 + (3.9-3.5)^2 + (1.7-1.4)^2} = \sqrt{0.09 + 0.16 + 0.09} = \sqrt{0.34} = 0.5831$$

In [ ]:
# ===== PERHITUNGAN JARAK KNN - IRIS =====
fitur = ['sepal_length', 'sepal_width', 'petal_length']

# Hitung jarak ke semua baris lain
hasil_jarak_iris = []
for i in range(len(iris_knn)):
    if i == target_idx:
        continue
    d = np.sqrt(((iris_knn.loc[target_idx, fitur] - iris_knn.loc[i, fitur]) ** 2).sum())
    hasil_jarak_iris.append((i, d, iris.loc[i, col_missing]))

# Urutkan berdasarkan jarak terkecil
hasil_jarak_iris = sorted(hasil_jarak_iris, key=lambda x: x[1])

# Ambil 3 tetangga terdekat
k = 3
tetangga_iris = hasil_jarak_iris[:k]

# Imputasi dengan rata-rata (KNN Regression)
imputasi_iris = np.mean([x[2] for x in tetangga_iris])

print("=" * 50)
print("IMPUTASI KNN - DATASET IRIS")
print("=" * 50)
print(f"Baris target: {target_idx}")
print(f"Kolom missing: {col_missing}")
print(f"Nilai asli: {nilai_asli_iris}")
print(f"\n3 tetangga terdekat:")
for t in tetangga_iris:
    data_row = iris.loc[t[0], fitur].values
    print(f"  Baris {t[0]}: jarak = {t[1]:.4f}, {col_missing} = {t[2]}, data = {data_row}")
print(f"\nHasil imputasi (rata-rata): {imputasi_iris:.4f}")
print(f"Nilai asli sebelum dihilangkan: {nilai_asli_iris}")

### Verifikasi Manual

3 tetangga terdekat yang ditemukan:

| Peringkat | Baris | Jarak | Data (SL, SW, PL) | petal_width |
|---|---|---|---|---|
| 1 | Baris 10 | 0.2828 | [5.4, 3.7, 1.5] | 0.2 |
| 2 | Baris 48 | 0.3000 | [5.3, 3.7, 1.5] | 0.2 |
| 3 | Baris 18 | 0.3162 | [5.7, 3.8, 1.7] | 0.3 |

**Imputasi:**
$$\hat{x} = \frac{0.2 + 0.2 + 0.3}{3} = \frac{0.7}{3} = 0.2333$$

In [ ]:
# ===== VISUALISASI: Tabel 10 Jarak Terkecil Iris =====
data_tabel = []
for t in hasil_jarak_iris[:10]:
    data_tabel.append([t[0], f"{t[1]:.4f}", t[2]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
tabel = ax.table(
    cellText=data_tabel,
    colLabels=['Baris', 'Jarak', 'petal_width'],
    loc='center'
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1, 1.5)

# Warnai 3 baris teratas (tetangga terdekat)
for row in range(1, 4):
    for col in range(3):
        tabel[row, col].set_facecolor('#d4edda')

plt.title('10 Jarak Terkecil - Dataset Iris (3 hijau = tetangga terpilih)', fontsize=11)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/tabel-jarak-iris.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== VISUALISASI: Tabel Hasil Imputasi Iris =====
iris_hasil = iris_knn.copy()
iris_hasil.loc[target_idx, 'petal_width'] = round(imputasi_iris, 4)

fig, ax = plt.subplots(figsize=(10, 2))
ax.axis('off')
tabel = ax.table(
    cellText=iris_hasil.loc[[target_idx]].values,
    colLabels=iris_hasil.columns,
    loc='center'
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1, 1.5)
plt.title(f'Baris {target_idx} Setelah Imputasi KNN (petal_width = {imputasi_iris:.4f})', fontsize=11)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/hasil-imputasi-iris.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Load Dataset Bank

In [ ]:
bank = pd.read_csv('bank.csv')
print(f"Dataset Bank: {bank.shape[0]} baris, {bank.shape[1]} kolom")
print(f"Kolom: {list(bank.columns)}")
bank.head(10)

---
## 8. Identifikasi Missing Value & Statistik Deskriptif — Bank

In [ ]:
# Cek missing value Bank
print("Missing value per kolom:")
print(bank.isnull().sum())
print(f"\nTotal missing value: {bank.isnull().sum().sum()}")
print("\n→ Dataset Bank TIDAK memiliki missing value.")
print("  Oleh karena itu, akan dibuat missing value BUATAN untuk simulasi KNN.")

In [ ]:
# Grafik cek missing value Bank
missing_count_bank = bank.isnull().sum()

plt.figure(figsize=(12, 4))
missing_count_bank.plot(kind='bar', color='steelblue')
plt.title('Jumlah Missing Value per Kolom - Bank (Sebelum Dibuat Buatan)', fontsize=12)
plt.xlabel('Kolom')
plt.ylabel('Jumlah Missing')
plt.ylim(0, 5)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/cek-missing-value-bank.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. KNN Imputation — Dataset Bank (Data Campuran)

### Konsep
Dataset Bank memiliki **data campuran** (numerik, ordinal, kategorikal). Jarak tidak bisa langsung dihitung dengan Euclidean saja.

### Tipe Variabel yang Digunakan:

| Kolom | Tipe Data | Keterangan |
|---|---|---|
| `age` | **Numerik** | Usia nasabah |
| `education` | **Ordinal** | primary < secondary < tertiary |
| `marital` | **Kategorikal** | married, single, divorced |
| `housing` | **Kategorikal** | yes, no |
| `loan` | **Kategorikal** | yes, no |
| `balance` | **Numerik (target)** | Kolom yang akan diimputasi |

### Langkah Jarak Campuran:
1. **Ordinal → numerik:** $z = \frac{r-1}{m-1}$
2. **Numerik → normalisasi:** Min-Max Scaling
3. **Jarak numerik+ordinal:** Euclidean Distance
4. **Jarak kategorikal:** $d_{kat} = \frac{P-M}{P}$
5. **Jarak total:** $d_{total} = d_{num+ord} + d_{kat}$

In [ ]:
# ===== MEMBUAT MISSING VALUE BUATAN - BANK =====
target_idx_bank = 5
col_missing_bank = 'balance'
nilai_asli_bank = bank.loc[target_idx_bank, col_missing_bank]

kolom_tampil = ['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'deposit']
kolom_tampil = [k for k in kolom_tampil if k in bank.columns]

print(f"Data asli baris {target_idx_bank}:")
print(bank.loc[[target_idx_bank], kolom_tampil])
print(f"\nNilai asli {col_missing_bank}: {nilai_asli_bank}")

bank_knn = bank.copy()
bank_knn.loc[target_idx_bank, col_missing_bank] = np.nan

print(f"\nData baris {target_idx_bank} setelah dibuat missing:")
print(bank_knn.loc[[target_idx_bank], kolom_tampil])

In [ ]:
# Tabel baris missing Bank
fig, ax = plt.subplots(figsize=(14, 2))
ax.axis('off')
tabel = ax.table(
    cellText=bank_knn.loc[[target_idx_bank], kolom_tampil].values,
    colLabels=kolom_tampil,
    loc='center'
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1, 1.5)
plt.title('Baris 5 dengan Missing Value Buatan (balance = NaN)', fontsize=11)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/baris-missing-bank.png', dpi=150, bbox_inches='tight')
plt.show()

### Langkah 1 — Konversi Ordinal ke Numerik

Kolom `education` dikonversi dengan rumus $z = \frac{r-1}{m-1}$:

| Level | r | Hasil |
|---|---|---|
| primary | 1 | 0 |
| secondary | 2 | 0.5 |
| tertiary | 3 | 1.0 |

### Langkah 2 — Normalisasi Numerik (Min-Max)

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

### Langkah 3 — Hitung Jarak Campuran

$$d_{total} = d_{num+ord} + d_{kat}$$

In [ ]:
# ===== PERHITUNGAN JARAK KNN - BANK (DATA CAMPURAN) =====

# Konversi ordinal
edu_order = {'primary': 1, 'secondary': 2, 'tertiary': 3}
m_edu = 3

def ord_norm(val):
    return (edu_order[val] - 1) / (m_edu - 1)

# Filter baris yang education-nya valid
bank_knn_filtered = bank_knn[bank_knn['education'].isin(edu_order.keys())].copy()
bank_knn_filtered = bank_knn_filtered.reset_index(drop=True)

# Cari ulang indeks target setelah reset
missing_idx = bank_knn_filtered[bank_knn_filtered[col_missing_bank].isna()].index[0]

# Normalisasi age (Min-Max)
age_min = bank_knn_filtered['age'].min()
age_max = bank_knn_filtered['age'].max()

def age_norm(val):
    return (val - age_min) / (age_max - age_min)

print(f"age_min = {age_min}, age_max = {age_max}")
print(f"Baris target setelah filter: {missing_idx}")
print(f"age baris target: {bank_knn_filtered.loc[missing_idx, 'age']} → normalized: {age_norm(bank_knn_filtered.loc[missing_idx, 'age']):.4f}")
print(f"education baris target: {bank_knn_filtered.loc[missing_idx, 'education']} → normalized: {ord_norm(bank_knn_filtered.loc[missing_idx, 'education']):.4f}")

# Hitung jarak campuran
cat_cols = ['marital', 'housing', 'loan']
hasil_jarak_bank = []

for i in range(len(bank_knn_filtered)):
    if i == missing_idx:
        continue

    # Jarak numerik + ordinal (Euclidean)
    d_numord = np.sqrt(
        (age_norm(bank_knn_filtered.loc[missing_idx, 'age']) - age_norm(bank_knn_filtered.loc[i, 'age']))**2 +
        (ord_norm(bank_knn_filtered.loc[missing_idx, 'education']) - ord_norm(bank_knn_filtered.loc[i, 'education']))**2
    )

    # Jarak kategorikal
    P = len(cat_cols)
    M = sum(bank_knn_filtered.loc[missing_idx, c] == bank_knn_filtered.loc[i, c] for c in cat_cols)
    d_cat = (P - M) / P

    # Jarak total
    d_total = d_numord + d_cat

    bal = bank_knn_filtered.loc[i, col_missing_bank]
    if pd.isna(bal):
        continue
    hasil_jarak_bank.append((i, d_total, d_numord, d_cat, bal))

# Urutkan berdasarkan jarak terkecil
hasil_jarak_bank = sorted(hasil_jarak_bank, key=lambda x: x[1])

# 3 tetangga terdekat
k = 3
tetangga_bank = hasil_jarak_bank[:k]
imputasi_bank = np.mean([x[4] for x in tetangga_bank])

print("\n" + "=" * 60)
print("IMPUTASI KNN - DATASET BANK (DATA CAMPURAN)")
print("=" * 60)
print(f"Baris target: {missing_idx}")
print(f"Kolom missing: {col_missing_bank}")
print(f"Nilai asli: {nilai_asli_bank}")
print(f"\n3 tetangga terdekat:")
for t in tetangga_bank:
    r = bank_knn_filtered.loc[t[0]]
    print(f"  Baris {t[0]}: d_total={t[1]:.4f} (d_numord={t[2]:.4f}, d_cat={t[3]:.4f}), balance={t[4]}")
    print(f"    age={r['age']}, edu={r['education']}, mar={r['marital']}, hou={r['housing']}, loan={r['loan']}")
print(f"\nHasil imputasi (rata-rata): {imputasi_bank:.2f}")
print(f"Nilai asli sebelum dihilangkan: {nilai_asli_bank}")

### Verifikasi Manual

3 tetangga terdekat yang ditemukan:

| Peringkat | Baris | d_num+ord | d_kat | d_total | balance |
|---|---|---|---|---|---|
| 1 | Baris 96 | 0.3243 | 0 | **0.3243** | 880 |
| 2 | Baris 21 | 0.0270 | 0.3333 | **0.3604** | 2067 |
| 3 | Baris 51 | 0.0811 | 0.3333 | **0.4144** | 517 |

**Perhatikan:**
- Baris 96: jarak numerik besar tapi kategorikal semua sama → total kecil
- Baris 21: jarak numerik kecil tapi ada 1 kategorikal beda → total lebih besar

**Imputasi:**
$$\hat{balance} = \frac{880 + 2067 + 517}{3} = \frac{3464}{3} = 1154.67$$

In [ ]:
# ===== VISUALISASI: Tabel 10 Jarak Terkecil Bank =====
data_tabel_bank = []
for t in hasil_jarak_bank[:10]:
    data_tabel_bank.append([
        t[0],
        f"{t[2]:.4f}",
        f"{t[3]:.4f}",
        f"{t[1]:.4f}",
        t[4]
    ])

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
tabel = ax.table(
    cellText=data_tabel_bank,
    colLabels=['Baris', 'd_num+ord', 'd_kat', 'd_total', 'balance'],
    loc='center'
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1, 1.5)

for row in range(1, 4):
    for col in range(5):
        tabel[row, col].set_facecolor('#d4edda')

plt.title('10 Jarak Terkecil - Dataset Bank (3 hijau = tetangga terpilih)', fontsize=11)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/tabel-jarak-bank.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== VISUALISASI: Tabel Hasil Imputasi Bank =====
bank_hasil = bank_knn_filtered.copy()
bank_hasil.loc[missing_idx, 'balance'] = round(imputasi_bank, 2)

kolom_tampil_bank = ['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'deposit']
kolom_tampil_bank = [k for k in kolom_tampil_bank if k in bank_hasil.columns]

fig, ax = plt.subplots(figsize=(14, 2))
ax.axis('off')
tabel = ax.table(
    cellText=bank_hasil.loc[[missing_idx], kolom_tampil_bank].values,
    colLabels=kolom_tampil_bank,
    loc='center'
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1, 1.5)
plt.title(f'Baris {missing_idx} Setelah Imputasi KNN (balance = {imputasi_bank:.2f})', fontsize=11)
plt.tight_layout()
plt.savefig('Assets/Pertemuan3/hasil-imputasi-bank.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Ringkasan

### Untuk Data Numerik (Iris)
1. Cek missing value → tidak ada → buat missing value buatan
2. Tentukan baris target dan kolom yang kosong
3. Hitung jarak Euclidean ke semua baris lain (tanpa kolom yang kosong)
4. Urutkan jarak dari terkecil
5. Ambil k=3 tetangga terdekat
6. Isi missing value = rata-rata nilai kolom target dari 3 tetangga

### Untuk Data Campuran (Bank)
1. Cek missing value → tidak ada → buat missing value buatan
2. Tentukan tipe setiap kolom: numerik, ordinal, atau kategorikal
3. Konversi ordinal ke numerik: $z = (r-1)/(m-1)$
4. Normalisasi kolom numerik: Min-Max Scaling
5. Hitung jarak numerik+ordinal: Euclidean
6. Hitung jarak kategorikal: $(P-M)/P$
7. Jumlahkan: $d_{total} = d_{num+ord} + d_{kat}$
8. Urutkan jarak dari terkecil
9. Ambil k=3 tetangga terdekat
10. Isi missing value = rata-rata dari 3 tetangga (KNN Regression)

### Hasil Imputasi
| Dataset | Kolom | Nilai Asli | Hasil Imputasi |
|---|---|---|---|
| Iris | petal_width | 0.4 | **0.2333** |
| Bank | balance | 0 | **1154.67** |